# Notebook 16 v2 — CGCS-constrained adaptive control

**Repo path:** `control-stack/16_cgcs_constrained_adaptive_control.ipynb`

This v2 pass fixes the weak/empty metric from v1 and makes the notebook more paper-ready:

- explicit 45° / `1 / sqrt(1² + 1²)` threshold definition
- CGCS margin and phase-error integral
- blocks-below-threshold metrics
- longest violation streak
- recovery-time summary
- ablation-style policies:
  - no compensation
  - moving average
  - joint Kalman
  - robust gated Kalman
  - naive adaptive gate Kalman
  - CGCS constrained adaptive gate Kalman
  - CGCS constrained adaptive joint Kalman
  - constrained MPC-lite
  - oracle

In [ ]:
# Notebook 16 v2 — setup

import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "16"
SLUG = "cgcs_constrained_adaptive_control"
TITLE = "CGCS-constrained adaptive control"

FIG_DIR = f"figures/{SLUG}"
RESULTS_DIR = "results"
DOCS_DIR = "docs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

np.random.seed(9423)

THRESHOLD = 1 / np.sqrt(1**2 + 1**2)

print(f"45° threshold = 1 / sqrt(1² + 1²) = {THRESHOLD:.6f}")
print(f"Figures -> {FIG_DIR}")
print(f"Results -> {RESULTS_DIR}")
print(f"Docs -> {DOCS_DIR}")

## 1. Synthetic calibration stream

The simulated control-stack stream includes smooth Ω drift, smooth B drift, isolated impulse noise, and two disturbance windows.

In [ ]:
# Synthetic calibration stream

n = 84
t = np.arange(n)

noise_burst_windows = [(28, 40), (54, 66)]
mismatch_windows = [(28, 40), (54, 66)]

def in_windows(i, windows):
    return any(a <= i <= b for a, b in windows)

noise_mask = np.array([in_windows(i, noise_burst_windows) for i in t])
mismatch_mask = np.array([in_windows(i, mismatch_windows) for i in t])

omega_true = (
    0.045 * np.sin(2*np.pi*t/18)
    + 0.020 * np.sin(2*np.pi*t/7 + 0.4)
    + 0.012 * np.cos(2*np.pi*t/31)
)
B_true = (
    0.004 + 0.00072*t
    + 0.008*np.sin(2*np.pi*t/26)
    + 0.004*np.sin(2*np.pi*t/9 + 0.8)
)

omega_meas = omega_true + np.random.normal(0, 0.006, n)
B_meas = B_true + np.random.normal(0, 0.004, n)

omega_meas[noise_mask] += np.random.normal(0, 0.030, noise_mask.sum())
B_meas[noise_mask] += np.random.normal(0, 0.020, noise_mask.sum())

outlier_blocks = np.array([17, 47, 72])
omega_meas[outlier_blocks] += np.array([0.17, -0.18, 0.11])
B_meas[outlier_blocks] += np.array([0.10, -0.08, 0.07])

pulse = np.linspace(0, 10, 450)

def response_curve(omega, B):
    phase = 1.55 * pulse + 10.5 * omega + 5.0 * B
    amp = 0.50 + 0.22 * np.tanh(8 * B)
    return 0.5 + amp * np.sin(phase)**2

target_curves = np.array([response_curve(omega_true[i], B_true[i]) for i in range(n)])

print("Synthetic stream ready:", n, "calibration blocks")

## 2. Utility functions

Policy comparison uses drift-estimate quality, response-level quality, and CGCS phase-lock quality.

In [ ]:
# Utilities

def moving_average(x, window=5):
    out = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i-window+1)
        out[i] = x[lo:i+1].mean()
    return out

def scalar_kalman(z, q=2e-5, r=5e-5):
    x = z[0]
    p = 1.0
    xs = []
    for zi in z:
        p = p + q
        k = p / (p + r)
        x = x + k * (zi - x)
        p = (1 - k) * p
        xs.append(x)
    return np.array(xs)

def joint_kalman(z_omega, z_B, q_omega=2e-5, q_B=1e-5, q_cross=1.2e-5, r_omega=5e-5, r_B=3e-5):
    z = np.vstack([z_omega, z_B]).T
    x = z[0].copy()
    P = np.eye(2)
    Q = np.array([[q_omega, q_cross], [q_cross, q_B]])
    R = np.diag([r_omega, r_B])
    H = np.eye(2)
    xs = []
    for zi in z:
        P = P + Q
        S = H @ P @ H.T + R
        K = P @ H.T @ np.linalg.inv(S)
        x = x + K @ (zi - H @ x)
        P = (np.eye(2) - K @ H) @ P
        xs.append(x.copy())
    return np.array(xs)

def robust_gated_kalman(z, q=2e-5, r=5e-5, gate=0.075):
    x = z[0]
    p = 1.0
    xs = []
    rejects = []
    for i, zi in enumerate(z):
        p = p + q
        innovation = zi - x
        if abs(innovation) > gate:
            rejects.append(i)
            xs.append(x)
            continue
        k = p / (p + r)
        x = x + k * innovation
        p = (1 - k) * p
        xs.append(x)
    return np.array(xs), np.array(rejects, dtype=int)

def adaptive_gate_kalman(z, q=2e-5, r=5e-5, base_gate=0.04, max_gate=0.22, decay=0.92, boost=1.45):
    x = z[0]
    p = 1.0
    gate = base_gate
    xs, gates, rejects = [], [], []
    for i, zi in enumerate(z):
        p = p + q
        innovation = zi - x
        if abs(innovation) > gate:
            rejects.append(i)
            gate = min(max_gate, max(gate*boost, abs(innovation)*1.05))
            xs.append(x)
        else:
            k = p / (p + r)
            x = x + k * innovation
            p = (1 - k) * p
            gate = max(base_gate, gate*decay)
            xs.append(x)
        gates.append(gate)
    return np.array(xs), np.array(gates), np.array(rejects, dtype=int)

def cgcs_constrained_adaptive_gate(z, q=2e-5, r=5e-5, base_gate=0.03, max_gate=0.09, decay=0.88, boost=1.20):
    x = z[0]
    p = 1.0
    gate = base_gate
    xs, gates, rejects = [], [], []
    for i, zi in enumerate(z):
        p = p + q
        innovation = zi - x
        if abs(innovation) > gate:
            rejects.append(i)
            gate = min(max_gate, max(gate*boost, abs(innovation)*0.58))
            x = x + 0.08 * np.sign(innovation) * gate
            xs.append(x)
        else:
            k = p / (p + r)
            x = x + k * innovation
            p = (1 - k) * p
            gate = max(base_gate, gate*decay)
            xs.append(x)
        gates.append(gate)
    return np.array(xs), np.array(gates), np.array(rejects, dtype=int)

def constrained_mpc_lite(est, command_bound=0.055, smooth=0.55):
    out = np.zeros_like(est)
    out[0] = est[0]
    for i in range(1, len(est)):
        desired = smooth*est[i] + (1-smooth)*out[i-1]
        step = np.clip(desired - out[i-1], -command_bound, command_bound)
        out[i] = out[i-1] + step
    return out

def cosine_similarity(a, b):
    aa = a - a.mean()
    bb = b - b.mean()
    denom = np.linalg.norm(aa) * np.linalg.norm(bb)
    if denom == 0:
        return 0.0
    return float(np.dot(aa, bb) / denom)

def response_rmse(curve, target):
    return float(np.sqrt(np.mean((curve - target)**2)))

def longest_true_streak(mask):
    best = cur = 0
    for v in mask:
        if v:
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best

def first_recovery_after(mask, start, min_good=3):
    for i in range(start, len(mask)-min_good+1):
        if not mask[i:i+min_good].any():
            return i - start
    return np.nan

def shade_windows(ax):
    for a,b in noise_burst_windows:
        ax.axvspan(a, b, alpha=0.16, label="noise burst" if a == noise_burst_windows[0][0] else None)
    for a,b in mismatch_windows:
        ax.axvspan(a, b, alpha=0.08, label="model mismatch" if a == mismatch_windows[0][0] else None)

## 3. Build policy estimates

In [ ]:
# Build policy estimates

omega_ma = moving_average(omega_meas, window=5)
B_ma = moving_average(B_meas, window=5)

omega_scalar = scalar_kalman(omega_meas, q=2e-5, r=6e-5)
B_scalar = scalar_kalman(B_meas, q=1e-5, r=3e-5)

joint_est = joint_kalman(omega_meas, B_meas, q_omega=3e-5, q_B=1.2e-5, q_cross=1.0e-5)
omega_joint, B_joint = joint_est[:,0], joint_est[:,1]

omega_robust, reject_robust = robust_gated_kalman(omega_meas, q=2e-5, r=6e-5, gate=0.075)
B_robust, reject_B_robust = robust_gated_kalman(B_meas, q=1e-5, r=3e-5, gate=0.055)

omega_adapt, omega_adapt_gate, reject_adapt = adaptive_gate_kalman(
    omega_meas, q=2e-5, r=6e-5, base_gate=0.04, max_gate=0.22
)
B_adapt, B_adapt_gate, reject_B_adapt = adaptive_gate_kalman(
    B_meas, q=1e-5, r=3e-5, base_gate=0.03, max_gate=0.12
)

omega_cgcs, omega_cgcs_gate, reject_cgcs = cgcs_constrained_adaptive_gate(
    omega_meas, q=2e-5, r=6e-5, base_gate=0.03, max_gate=0.09
)
B_cgcs, B_cgcs_gate, reject_B_cgcs = cgcs_constrained_adaptive_gate(
    B_meas, q=1e-5, r=3e-5, base_gate=0.025, max_gate=0.07
)

omega_cgcs_joint = 0.70 * omega_cgcs + 0.30 * omega_joint
B_cgcs_joint = 0.70 * B_cgcs + 0.30 * B_joint

omega_mpc = constrained_mpc_lite(omega_cgcs_joint, command_bound=0.055, smooth=0.62)
B_mpc = constrained_mpc_lite(B_cgcs_joint, command_bound=0.040, smooth=0.62)

policies = {
    "none": (omega_meas, B_meas),
    "moving_average": (omega_ma, B_ma),
    "joint_kalman": (omega_joint, B_joint),
    "robust_gated_kalman": (omega_robust, B_robust),
    "naive_adaptive_gate_kalman": (omega_adapt, B_adapt),
    "cgcs_constrained_adaptive_gate_kalman": (omega_cgcs, B_cgcs),
    "cgcs_constrained_adaptive_joint_kalman": (omega_cgcs_joint, B_cgcs_joint),
    "constrained_mpc": (omega_mpc, B_mpc),
    "oracle": (omega_true, B_true),
}

print("Policies:", list(policies.keys()))
print("robust rejects:", len(reject_robust), "adaptive rejects:", len(reject_adapt), "cgcs rejects:", len(reject_cgcs))

## 4. Response metrics and CGCS metrics

In [ ]:
# Response and CGCS metrics

policy_curves = {}
rmse_by_policy = {}
cos_by_policy = {}
margin_by_policy = {}

for name, (om, bb) in policies.items():
    curves = np.array([response_curve(om[i], bb[i]) for i in range(n)])
    policy_curves[name] = curves
    rmse_by_policy[name] = np.array([response_rmse(curves[i], target_curves[i]) for i in range(n)])
    cos_by_policy[name] = np.array([cosine_similarity(curves[i], target_curves[i]) for i in range(n)])
    margin_by_policy[name] = cos_by_policy[name] - THRESHOLD

summary_rows = []
for name in policies:
    rmse = rmse_by_policy[name]
    cosv = cos_by_policy[name]
    failures = cosv < THRESHOLD
    phase_integral = float(np.sum(np.maximum(0, 1 - cosv)))
    below = int(failures.sum())
    longest = int(longest_true_streak(failures))
    rec_times = []
    for a,b in noise_burst_windows:
        rt = first_recovery_after(failures, b+1, min_good=3)
        if not np.isnan(rt):
            rec_times.append(rt)
    mean_recovery = float(np.mean(rec_times)) if rec_times else 0.0
    summary_rows.append({
        "policy": name,
        "mean_rmse": float(rmse.mean()),
        "max_rmse": float(rmse.max()),
        "mean_cosine": float(cosv.mean()),
        "min_cosine": float(cosv.min()),
        "blocks_below_threshold": below,
        "longest_violation_streak": longest,
        "phase_error_integral": phase_integral,
        "mean_recovery_blocks": mean_recovery,
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["blocks_below_threshold", "phase_error_integral", "mean_rmse"],
    ascending=[True, True, True]
)

display(summary_df)
summary_df.to_csv(f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_policy_summary.csv", index=False)

summary_json = {
    "notebook_id": NOTEBOOK_ID,
    "slug": SLUG,
    "title": TITLE,
    "threshold": float(THRESHOLD),
    "threshold_label": "1 / sqrt(1^2 + 1^2)",
    "noise_burst_windows": noise_burst_windows,
    "model_mismatch_windows": mismatch_windows,
    "best_policy_by_threshold_blocks": summary_df.iloc[0]["policy"],
    "policies": summary_rows,
}
with open(f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_summary.json", "w") as f:
    json.dump(summary_json, f, indent=2)

print("Saved results:")
print(f"- {RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_policy_summary.csv")
print(f"- {RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_summary.json")

In [ ]:
phase_df = summary_df.sort_values("phase_error_integral")
plt.figure(figsize=(12, 6))
plt.bar(phase_df["policy"], phase_df["phase_error_integral"])
plt.title("CGCS-constrained adaptive control: phase-error integral")
plt.ylabel("Σ max(0, 1 - cosine)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_phase_error_integral.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
plt.plot(t, omega_true, label="true Ω drift", linewidth=2)
plt.plot(t, omega_meas, "o-", label="measured Ω", alpha=0.65)
for name in ["joint_kalman", "robust_gated_kalman", "naive_adaptive_gate_kalman", "cgcs_constrained_adaptive_gate_kalman", "cgcs_constrained_adaptive_joint_kalman", "constrained_mpc"]:
    plt.plot(t, policies[name][0], label=name)
plt.title("CGCS-constrained adaptive control: Ω tracking")
plt.xlabel("calibration block")
plt.ylabel("Ω drift estimate / command")
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_omega_tracking.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
plt.plot(t, B_true, label="true B drift", linewidth=2)
plt.plot(t, B_meas, "o-", label="measured B", alpha=0.65)
for name in ["joint_kalman", "robust_gated_kalman", "naive_adaptive_gate_kalman", "cgcs_constrained_adaptive_gate_kalman", "cgcs_constrained_adaptive_joint_kalman", "constrained_mpc"]:
    plt.plot(t, policies[name][1], label=name)
plt.title("CGCS-constrained adaptive control: B tracking")
plt.xlabel("calibration block")
plt.ylabel("B drift estimate / command")
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_b_tracking.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
for name in policies:
    if name == "oracle":
        continue
    plt.plot(t, rmse_by_policy[name], "o-", label=name, alpha=0.90)
plt.plot(t, rmse_by_policy["oracle"], "o-", label="oracle", linewidth=2)
plt.title("CGCS-constrained adaptive control: response RMSE comparison")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target response")
plt.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_response_rmse_comparison.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
plt.axhline(THRESHOLD, linestyle="--", label="45° threshold")
for name in policies:
    plt.plot(t, cos_by_policy[name], "o-", label=name, alpha=0.90)
plt.title("CGCS-constrained adaptive control: phase-lock stability")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.ylim(0.55, 1.02)
plt.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_cgcs_stability_comparison.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
ax = plt.gca()
shade_windows(ax)
plt.axhline(0, linestyle="--", label="threshold margin = 0")
for name in policies:
    if name == "oracle":
        continue
    plt.plot(t, margin_by_policy[name], "o-", label=name, alpha=0.90)
plt.title("CGCS-constrained adaptive control: threshold margin")
plt.xlabel("calibration block")
plt.ylabel("CGCS margin above threshold")
plt.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_threshold_margin.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
below_df = summary_df.sort_values("blocks_below_threshold")
plt.figure(figsize=(12, 6))
plt.bar(below_df["policy"], below_df["blocks_below_threshold"])
plt.title("CGCS-constrained adaptive control: blocks below 45° threshold")
plt.ylabel("blocks below threshold")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_blocks_below_threshold.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
policy_order = list(summary_df["policy"])
failure_matrix = np.vstack([(cos_by_policy[name] < THRESHOLD).astype(int) for name in policy_order])

plt.figure(figsize=(15, 6))
plt.imshow(failure_matrix, aspect="auto", interpolation="nearest")
plt.yticks(np.arange(len(policy_order)), policy_order)
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.title("CGCS-constrained adaptive control: failure-event map")
cbar = plt.colorbar()
cbar.set_label("failure: cosθ < threshold")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_failure_event_map.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
rank_df = summary_df.sort_values("mean_rmse")
x = np.arange(len(rank_df))
width = 0.38

plt.figure(figsize=(13, 6))
plt.bar(x - width/2, rank_df["mean_rmse"], width, label="mean RMSE")
plt.bar(x + width/2, rank_df["max_rmse"], width, label="max RMSE")
plt.xticks(x, rank_df["policy"], rotation=25, ha="right")
plt.title("CGCS-constrained adaptive control: policy ranking")
plt.ylabel("response RMSE")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_policy_ranking_summary.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
ax = plt.gca()
shade_windows(ax)
plt.plot(t, omega_adapt_gate, label="naive adaptive Ω gate")
plt.plot(t, omega_cgcs_gate, label="CGCS constrained adaptive Ω gate")
plt.axhline(0.075, linestyle="--", label="fixed robust Ω gate")
plt.title("CGCS-constrained adaptive control: adaptive gate trace")
plt.xlabel("calibration block")
plt.ylabel("Ω innovation gate")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_adaptive_gate_trace.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(15, 4))
plt.scatter(reject_robust, np.zeros_like(reject_robust), marker="x", s=90, label="robust_gated_kalman")
plt.scatter(reject_adapt, np.ones_like(reject_adapt), marker="x", s=90, label="naive_adaptive_gate_kalman")
plt.scatter(reject_cgcs, np.ones_like(reject_cgcs)*2, marker="x", s=90, label="cgcs_constrained_adaptive_gate_kalman")
plt.yticks([0,1,2], ["robust_gated_kalman", "naive_adaptive_gate_kalman", "cgcs_constrained_adaptive_gate_kalman"])
plt.title("CGCS-constrained adaptive control: rejection events")
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_rejection_events.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
candidate = "cgcs_constrained_adaptive_joint_kalman"
worst_block = int(np.argmax(rmse_by_policy[candidate]))

plt.figure(figsize=(14, 6))
plt.plot(pulse, target_curves[worst_block], label="target", linewidth=3)
for name in ["none", "moving_average", "joint_kalman", "robust_gated_kalman", "naive_adaptive_gate_kalman", "cgcs_constrained_adaptive_joint_kalman", "constrained_mpc", "oracle"]:
    plt.plot(pulse, policy_curves[name][worst_block], label=name)
plt.title(f"CGCS-constrained adaptive control: worst-case block {worst_block}")
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_worst_case_block_comparison.png", dpi=220, bbox_inches="tight")
plt.show()

print("Worst block for", candidate, "=", worst_block)

## 5. Export markdown report

The markdown uses the existing repo convention:

`![Figure](../figures/<folder>/<file>.png)`

In [ ]:
# Export docs markdown with repo-relative image paths

figures = [
    ("Phase-error integral", "phase_error_integral", "Lower is better; integrates cosine-alignment error across the calibration stream."),
    ("Ω tracking", "omega_tracking", "CGCS-constrained estimates avoid unconstrained adaptive spikes while tracking drift."),
    ("B tracking", "b_tracking", "B tracking checks coupled behavior across slow drift and disturbance windows."),
    ("Response RMSE comparison", "response_rmse_comparison", "Response-level RMSE captures target-behavior mismatch after estimation/control."),
    ("CGCS stability comparison", "cgcs_stability_comparison", "Cosine similarity against target response makes the 45° phase-lock gate visible."),
    ("Threshold margin", "threshold_margin", "Margin is cosine similarity minus the 45° threshold."),
    ("Blocks below threshold", "blocks_below_threshold", "Counts actual phase-lock violations; this fixes the empty v1 metric."),
    ("Failure-event map", "failure_event_map", "Binary phase-lock failure map by policy and block."),
    ("Policy ranking summary", "policy_ranking_summary", "RMSE ranking and CGCS ranking are related but not identical."),
    ("Adaptive gate trace", "adaptive_gate_trace", "CGCS-bound adaptive gate prevents naive adaptive gate expansion."),
    ("Rejection events", "rejection_events", "Innovation rejections show where robust and adaptive gates intervene."),
    ("Worst-case block comparison", "worst_case_block_comparison", "Worst-block response comparison shows phase alignment directly."),
]

doc_path = f"{DOCS_DIR}/{NOTEBOOK_ID}_{SLUG}.md"

best_threshold = summary_df.iloc[0]["policy"]
best_rmse = summary_df.sort_values("mean_rmse").iloc[0]["policy"]

lines = []
lines.append(f"# {TITLE}\n")
lines.append("Notebook 16 v2 evaluates adaptive control under noise bursts and model mismatch using CGCS/45° phase-lock metrics.\n")
lines.append("## Key constants\n")
lines.append(f"- 45° threshold: `1 / sqrt(1^2 + 1^2) = {THRESHOLD:.6f}`\n")
lines.append("- CGCS margin: `cosine_similarity - threshold`\n")
lines.append("- Failure event: `cosine_similarity < threshold`\n")
lines.append("\n## Results summary\n")
lines.append(f"- Best policy by threshold violations: `{best_threshold}`\n")
lines.append(f"- Best policy by mean RMSE: `{best_rmse}`\n")
lines.append(f"- Results CSV: `../results/{NOTEBOOK_ID}_{SLUG}_policy_summary.csv`\n")
lines.append(f"- Results JSON: `../results/{NOTEBOOK_ID}_{SLUG}_summary.json`\n")
lines.append("\n## Policy summary table\n")
lines.append(summary_df.to_markdown(index=False))
lines.append("\n\n## Figures\n")
for title, fname, caption in figures:
    file = f"{NOTEBOOK_ID}_{SLUG}_{fname}.png"
    lines.append(f"\n### {title}\n")
    lines.append(f"\n![{title}](../figures/{SLUG}/{file})\n")
    lines.append(f"\n{caption}\n")

with open(doc_path, "w") as f:
    f.write("\n".join(lines))

print(f"Saved markdown: {doc_path}")

In [ ]:
# Zip Notebook 16 v2 outputs for Colab download

zip_name = f"{NOTEBOOK_ID}_{SLUG}_v2_outputs.zip"

files_to_zip = [
    f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_policy_summary.csv",
    f"{RESULTS_DIR}/{NOTEBOOK_ID}_{SLUG}_summary.json",
    f"{DOCS_DIR}/{NOTEBOOK_ID}_{SLUG}.md",
]

for _, fname, _ in figures:
    files_to_zip.append(f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_{fname}.png")

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in files_to_zip:
        if os.path.exists(path):
            z.write(path)
        else:
            print("[warn] missing:", path)

print("Created:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception as e:
    print("Colab download skipped or unavailable:", e)
    print("Download manually from runtime:", zip_name)